#**PTQ Static**

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = make_moons(n_samples=5000, noise=0.3, random_state=42)

X = StandardScaler().fit_transform(X)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Original FP32 Model
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))

model_fp32 = MLP()

optimizer = torch.optim.Adam(model_fp32.parameters(), lr=0.01)

loss_fn = nn.BCELoss()

for epoch in range(2000):
    model_fp32.train()
    optimizer.zero_grad()
    out = model_fp32(X_train)
    loss = loss_fn(out, y_train)
    loss.backward()
    optimizer.step()

  # Accuracy
def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X)
        preds = (preds > 0.5).float()
        return (preds == y).float().mean().item()

print("\nFP32 Accuracy:", accuracy(model_fp32, X_test, y_test))


FP32 Accuracy: 0.9150000214576721


In [34]:
def quantize_tensor(t, num_bits=8):
    qmin = -2**(num_bits - 1) #-128
    qmax = 2**(num_bits - 1) - 1 #127
    min_val, max_val = t.min(), t.max()
    scale = (max_val - min_val) / (qmax - qmin + 1e-8)
    zp = torch.round(-min_val / scale).to(torch.int32)
    q_t = torch.clamp(torch.round(t / scale) + zp, qmin, qmax).to(torch.int8)
    return q_t, scale, zp

def dequantize_tensor(q_t, scale, zp):
    return (q_t.float() - zp) * scale

In [35]:
# Calibration: Get activation ranges
def get_activation_min_max(model, X_sample):
    act_ranges = {}
    hooks = []

    def register_hook(name):
        def hook_fn(module, input, output):
            act_min = output.min().item()
            act_max = output.max().item()
            act_ranges[name] = (act_min, act_max)
        return hook_fn

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(register_hook(name)))

    with torch.no_grad():
        model.eval()
        model(X_sample)

    for h in hooks:
        h.remove()

    return act_ranges

In [36]:
X_calib = X_train[:100]
act_ranges = get_activation_min_max(model_fp32, X_calib)
act_ranges

{'fc1': (-4.078413009643555, 5.018320560455322),
 'fc2': (-2.434211015701294, 7.994430065155029),
 'fc3': (-15.668591499328613, 8.965485572814941)}

In [37]:
class QuantizedMLP(nn.Module):
    def __init__(self, fp_model, act_ranges):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

        # Quantize weights
        with torch.no_grad():
            for (name_fp, param_fp), (name_q, param_q) in zip(fp_model.named_parameters(), self.named_parameters()):
                q_param, scale, zp = quantize_tensor(param_fp.data)
                dq_param = dequantize_tensor(q_param, scale, zp)
                param_q.data.copy_(dq_param)

        # Store activation scales
        self.act_scales = {}
        for name, (amin, amax) in act_ranges.items():
            scale = (amax - amin) / 255.0
            zp = round(-amin / scale) if scale > 0 else 0
            self.act_scales[name] = (scale, zp)

    def quant_act(self, x, layer_name):
        scale, zp = self.act_scales[layer_name]
        q_x = torch.clamp(torch.round(x / scale) + zp, 0, 255).to(torch.uint8)
        dq_x = (q_x.float() - zp) * scale
        return dq_x

    def forward(self, x):
        x = F.relu(self.quant_act(self.fc1(x), 'fc1'))
        x = F.relu(self.quant_act(self.fc2(x), 'fc2'))
        return torch.sigmoid(self.fc3(x))

In [38]:
model_static_ptq = QuantizedMLP(model_fp32, act_ranges)
print("STATIC PTQ Accuracy:", accuracy(model_static_ptq, X_test, y_test))

STATIC PTQ Accuracy: 0.47600001096725464


#**QAT** (Quantization Aware Training)

In [39]:
# QAT-compatible Model
class QATMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        x = self.dequant(x)
        return x

In [40]:
model = QATMLP()

In [41]:
torch.quantization.get_default_qat_qconfig('fbgemm')

QConfig(activation=functools.partial(<class 'torch.ao.quantization.fake_quantize.FusedMovingAvgObsFakeQuantize'>, observer=<class 'torch.ao.quantization.observer.MovingAverageMinMaxObserver'>, quant_min=0, quant_max=255, reduce_range=True){}, weight=functools.partial(<class 'torch.ao.quantization.fake_quantize.FusedMovingAvgObsFakeQuantize'>, observer=<class 'torch.ao.quantization.observer.MovingAveragePerChannelMinMaxObserver'>, quant_min=-128, quant_max=127, dtype=torch.qint8, qscheme=torch.per_channel_symmetric){})

In [42]:
# Define quantization config
model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')

# Prepare QAT model (insert fake quant nodes)
torch.quantization.prepare_qat(model, inplace=True)

/tmp/ipykernel_11903/741060896.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare_qat(model, inplace=True)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of 

QATMLP(
  (quant): QuantStub(
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
      (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
    )
  )
  (fc1): Linear(
    in_features=2, out_features=16, bias=True
    (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.qint8, quant_min=-128, quant_max=127, qscheme=torch.per_channel_symmetric, reduce_range=False
      (activation_post_process): MovingAveragePerChannelMinMaxObserver(min_val=tensor([]), max_val=tensor([]))
    )
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observe

In [43]:
# Optimizer + Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

# Train with Fake Quantization
for epoch in range(2000):
    model.train()
    optimizer.zero_grad()
    out = model(X_train)
    loss = loss_fn(out, y_train)
    loss.backward()
    optimizer.step()
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 0.6929
Epoch 500 | Loss: 0.2571
Epoch 1000 | Loss: 0.2092
Epoch 1500 | Loss: 0.2052


In [44]:
print("Accuracy before convert():", accuracy(model, X_test, y_test))

Accuracy before convert(): 0.9139999747276306


In [45]:
model_int8 = torch.quantization.convert(model.eval(), inplace=False)

print("QAT INT8 Accuracy:", accuracy(model_int8, X_test, y_test))

QAT INT8 Accuracy: 0.9150000214576721


/tmp/ipykernel_11903/2714054867.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.convert(model.eval(), inplace=False)


#**Dynamic PTQ**

In [46]:
class BigMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, 16)
        self.fc6 = nn.Linear(16, 8)
        self.fc7 = nn.Linear(8, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        x = F.relu(self.fc6(x))
        return torch.sigmoid(self.fc7(x))

In [47]:
model_fp32 = BigMLP()

In [48]:
# Optimizer + Loss
optimizer = torch.optim.Adam(model_fp32.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

# Train with Fake Quantization
for epoch in range(2000):
    model_fp32.train()
    optimizer.zero_grad()
    out = model_fp32(X_train)
    loss = loss_fn(out, y_train)
    loss.backward()
    optimizer.step()
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 0.6939
Epoch 500 | Loss: 0.1882
Epoch 1000 | Loss: 0.1717
Epoch 1500 | Loss: 0.1824


In [49]:
print("\nFP32 Accuracy:", accuracy(model_fp32, X_test, y_test))


FP32 Accuracy: 0.9070000052452087


In [50]:
from torch.quantization import quantize_dynamic

In [51]:
model_int8 = quantize_dynamic(model_fp32, {nn.Linear}, dtype=torch.qint8)

/tmp/ipykernel_11903/1760000380.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = quantize_dynamic(model_fp32, {nn.Linear}, dtype=torch.qint8)


In [52]:
model_int8

BigMLP(
  (fc1): DynamicQuantizedLinear(in_features=2, out_features=128, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  (fc2): DynamicQuantizedLinear(in_features=128, out_features=64, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  (fc3): DynamicQuantizedLinear(in_features=64, out_features=64, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  (fc4): DynamicQuantizedLinear(in_features=64, out_features=32, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  (fc5): DynamicQuantizedLinear(in_features=32, out_features=16, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  (fc6): DynamicQuantizedLinear(in_features=16, out_features=8, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  (fc7): DynamicQuantizedLinear(in_features=8, out_features=1, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
)

In [53]:
print("\nFP32 Accuracy:", accuracy(model_int8, X_test, y_test))


FP32 Accuracy: 0.9049999713897705


In [56]:
print(model_fp32.named_parameters)

<bound method Module.named_parameters of BigMLP(
  (fc1): Linear(in_features=2, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=64, bias=True)
  (fc4): Linear(in_features=64, out_features=32, bias=True)
  (fc5): Linear(in_features=32, out_features=16, bias=True)
  (fc6): Linear(in_features=16, out_features=8, bias=True)
  (fc7): Linear(in_features=8, out_features=1, bias=True)
)>


In [58]:
new_model = BigMLP()

In [59]:
with torch.no_grad():
    for (name_fp, param_fp), (name_q, param_q) in zip(model_fp32.named_parameters(), new_model.named_parameters()):

        q_param, scale, zp = quantize_tensor(param_fp.data)

        dq_param = dequantize_tensor(q_param, scale, zp)

        param_q.data.copy_(dq_param)

In [60]:
print("\nFP32 Accuracy:", accuracy(new_model, X_test, y_test))


FP32 Accuracy: 0.47600001096725464
